# 02. 벡터 데이터 다루기 — geopandas·shapely

**벡터(vector)** 데이터는 점·선·면(폴리곤)으로 지리 객체를 표현합니다 (예: 건물, 도로, 행정구역).
`shapely`로 도형을 만들고, `geopandas`로 표 형태로 관리하며 공간 연산을 수행합니다.

1. shapely로 도형 만들기
2. GeoDataFrame 구성
3. 공간 연산 (면적·버퍼·교집합)
4. 좌표계 변환 (CRS)

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, Polygon
import matplotlib.pyplot as plt

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
print('geopandas', gpd.__version__)

## 1. shapely 도형

점(Point)과 면(Polygon)을 만듭니다. shapely는 기하 연산(면적·거리·교집합 등)을 제공합니다.

In [ ]:
poly_a = Polygon([(0, 0), (4, 0), (4, 4), (0, 4)])
poly_b = Polygon([(2, 2), (6, 2), (6, 6), (2, 6)])
print('A 면적:', poly_a.area)
print('A와 B 교집합 면적:', poly_a.intersection(poly_b).area)
print('점 (1,1)이 A 안에 있는가:', poly_a.contains(Point(1, 1)))

## 2. GeoDataFrame

geopandas의 GeoDataFrame은 pandas DataFrame에 **geometry 열**을 더한 것입니다. 속성과 도형을 함께 관리합니다.

In [ ]:
gdf = gpd.GeoDataFrame({
    'name': ['지역A', '지역B'],
    'population': [1200, 3400],
    'geometry': [poly_a, poly_b],
}, crs='EPSG:4326')

print(gdf)
print('\n각 지역 면적:')
print(gdf.geometry.area)

## 3. 공간 연산과 시각화

버퍼(buffer, 일정 거리 확장)와 교집합 같은 공간 연산을 수행하고 시각화합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
gdf.plot(ax=axes[0], column='name', alpha=0.5, edgecolor='black', legend=True)
axes[0].set_title('두 지역')

# 교집합 영역
inter = gpd.GeoSeries([poly_a.intersection(poly_b)], crs='EPSG:4326')
gdf.plot(ax=axes[1], alpha=0.3, edgecolor='black')
inter.plot(ax=axes[1], color='red')
axes[1].set_title('교집합 (빨강)')
plt.tight_layout(); plt.show()

## 4. 좌표계 변환

지리 데이터는 분석 목적에 따라 좌표계(CRS)를 바꿔야 합니다.
경위도(EPSG:4326)는 면적 계산에 부적합하므로, 미터 단위 투영좌표계(예: UTM)로 변환합니다.

In [ ]:
gdf_utm = gdf.to_crs('EPSG:32618')   # UTM Zone 18N (미터 단위)
print('변환 후 CRS:', gdf_utm.crs)
print('미터 단위 면적(제곱미터):')
print(gdf_utm.geometry.area)

### 정리

- shapely로 도형을 만들고 geopandas로 속성과 함께 관리하며 공간 연산을 수행했습니다.
- 좌표계 변환(`to_crs`)으로 올바른 단위의 면적을 계산했습니다 — 지리 분석에서 CRS 관리는 필수입니다.
- 다음 노트북에서는 래스터와 결합해 식생지수(NDVI)를 계산합니다.